### Import Libraries and Setup

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, mean_squared_error,
    mean_absolute_error
)
import xgboost as xgb
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from imblearn.over_sampling import SMOTENC
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Create artifacts directory
ARTIFACTS_DIR = 'model_artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print("✓ Libraries imported successfully")
print(f"✓ Artifacts directory created: {ARTIFACTS_DIR}")



✓ Libraries imported successfully
✓ Artifacts directory created: model_artifacts


### Load Dataset

In [22]:
# Load the predictive maintenance synthesized dataset
# Replace with actual file path
df = pd.read_csv('preprocessed_data\predictive_maintenance_synthesized.csv')

print("\n" + "="*70)
print("DATASET OVERVIEW")
print("="*70)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFailure type distribution:\n{df['failure_type'].value_counts()}")


DATASET OVERVIEW
Dataset shape: (57912, 9)

Column names:
['engine_type', 'air_temp', 'process_temp', 'rpm', 'torque_nm', 'tool_wear', 'failure_type', 'product_id', 'timestamp']

First few rows:
  engine_type  air_temp  process_temp       rpm  torque_nm  tool_wear  \
0           H -0.802401     -0.745157  1.356707  -1.613785  -1.318927   
1           H -0.702409     -0.610355 -0.645801   0.432675  -1.240374   
2           H -0.602417     -0.542955 -1.298429   1.455905  -0.910450   
3           H -0.552421     -0.475554 -0.913546   0.272168  -0.784764   
4           H -0.452429     -0.408153  1.518470  -1.543563  -0.486261   

  failure_type product_id            timestamp  
0   No Failure     H29424  2024-01-11 16:00:00  
1   No Failure     H29425  2024-03-20 11:00:00  
2   No Failure     H29432  2024-02-22 17:00:00  
3   No Failure     H29434  2024-01-11 15:00:00  
4   No Failure     H29441  2024-01-25 07:00:00  

Data types:
engine_type      object
air_temp        float64
process_te

### Advanced Preprocessing - Encoding & Feature Engineering

In [23]:
print("\n" + "="*70)
print("PREPROCESSING: ENCODING & FEATURE ENGINEERING")
print("="*70)

# Create a copy for processing
df_processed = df.copy()

# Debug: Check initial state
print(f"\n🔍 INITIAL DATASET:")
print(f"  Shape: {df_processed.shape}")
print(f"  Columns: {df_processed.columns.tolist()}")
print(f"  Initial NaN count: {df_processed.isnull().sum().sum()}")

# CRITICAL: Remove rows with NaN values FIRST (before any processing)
initial_len = len(df_processed)
df_processed = df_processed.dropna()
dropped_initial = initial_len - len(df_processed)
print(f"  Dropped {dropped_initial} rows with NaN from original data")
print(f"  Remaining rows: {len(df_processed)}")

if len(df_processed) == 0:
    raise ValueError(
        "\n❌ ERROR: Original CSV has all NaN values or is empty!\n"
        f"Check file: preprocessed_data\\predictive_maintenance_synthesized.csv"
    )

# 1. ENCODING
le_engine = LabelEncoder()
df_processed['engine_type_encoded'] = le_engine.fit_transform(df_processed['engine_type'])

le_failure = LabelEncoder()
df_processed['failure_type_encoded'] = le_failure.fit_transform(df_processed['failure_type'])

# Save encoders
joblib.dump(le_engine, os.path.join(ARTIFACTS_DIR, 'encoder_engine_type.joblib'))
joblib.dump(le_failure, os.path.join(ARTIFACTS_DIR, 'encoder_failure_type.joblib'))

print("\n✓ Label encoders saved")
print(f"  Engine types: {le_engine.classes_}")
print(f"  Failure types: {le_failure.classes_}")

# 2. CREATE product_id - CRITICAL FIX: Use np.arange() instead of index
if 'product_id' not in df.columns or df_processed['product_id'].nunique() == len(df_processed):
    print("\n⚠️  Creating proper product IDs based on sequential grouping...")
    
    # FIXED: Use np.arange to create sequential numbering
    TIMESTEPS_PER_PRODUCT = 100
    row_numbers = np.arange(len(df_processed))
    df_processed['product_id'] = row_numbers // TIMESTEPS_PER_PRODUCT
    
    print(f"  ✓ Created {df_processed['product_id'].nunique()} unique product IDs")
    
    # Verify distribution
    product_counts = df_processed.groupby('product_id').size()
    print(f"  Timesteps per product:")
    print(f"    Mean: {product_counts.mean():.1f}")
    print(f"    Min: {product_counts.min()}")
    print(f"    Max: {product_counts.max()}")
    print(f"    Products with ≥10 timesteps: {(product_counts >= 10).sum()} / {df_processed['product_id'].nunique()}")
    
    # Sample distribution
    print(f"\n  Sample product_id distribution:")
    sample_pids = df_processed['product_id'].iloc[:300:10]
    print(f"    First 30 rows (every 10th): {sample_pids.tolist()}")
else:
    print("\n✓ product_id column already exists and properly distributed")

# 3. CREATE RUL (Remaining Useful Life) COLUMN
print("\n### Creating RUL (Remaining Useful Life) ###")

if 'rul' not in df_processed.columns:
    print("⚠️  'rul' column not found - Creating synthetic RUL based on failure_type...")
    
    # Strategy 1: Simple RUL based on failure type
    # Assign different RUL ranges for different failure types
    rul_mapping = {
        'No Failure': (180, 365),           # 6 months to 1 year
        'Tool Wear Failure': (30, 90),      # 1-3 months
        'Heat Dissipation Failure': (60, 120),  # 2-4 months
        'Power Failure': (45, 105),         # 1.5-3.5 months
        'Overstrain Failure': (15, 60),     # 0.5-2 months
        'Random Failures': (90, 180)        # 3-6 months
    }
    
    # Create RUL column
    df_processed['rul'] = 0
    
    for failure_type, (min_rul, max_rul) in rul_mapping.items():
        mask = df_processed['failure_type'] == failure_type
        count = mask.sum()
        if count > 0:
            # Generate random RUL within the range
            df_processed.loc[mask, 'rul'] = np.random.randint(
                min_rul, max_rul + 1, size=count
            )
    
    # Strategy 2: Adjust RUL based on sensor readings (optional refinement)
    # Higher tool_wear = lower RUL
    high_wear_mask = df_processed['tool_wear'] > df_processed['tool_wear'].quantile(0.75)
    df_processed.loc[high_wear_mask, 'rul'] = (df_processed.loc[high_wear_mask, 'rul'] * 0.7).astype(int)
    
    # Higher temperatures = lower RUL
    high_temp_mask = df_processed['process_temp'] > df_processed['process_temp'].quantile(0.75)
    df_processed.loc[high_temp_mask, 'rul'] = (df_processed.loc[high_temp_mask, 'rul'] * 0.8).astype(int)
    
    # Ensure minimum RUL of 1 day
    df_processed['rul'] = df_processed['rul'].clip(lower=1)
    
    print(f"  ✓ RUL column created")
    print(f"  RUL statistics:")
    print(f"    Mean: {df_processed['rul'].mean():.1f} days")
    print(f"    Median: {df_processed['rul'].median():.0f} days")
    print(f"    Min: {df_processed['rul'].min()} days")
    print(f"    Max: {df_processed['rul'].max()} days")
    print(f"\n  RUL by failure type:")
    print(df_processed.groupby('failure_type')['rul'].agg(['mean', 'min', 'max']).round(1))
else:
    print("✓ RUL column already exists")

# 4. FEATURE ENGINEERING - SKIP LAGGING
print("\n### Feature Engineering ###")
print("⚠️  SKIPPING lag features to preserve data")
LAG_PERIODS = []  # NO LAGGING

# Identify sensor columns (no lagging)
exclude_cols_list = [
    'product_id', 'engine_type', 'failure_type', 
    'engine_type_encoded', 'failure_type_encoded', 'rul', 
    'timestamp', 'cycle'
]
sensor_cols = [col for col in df_processed.columns if 
               col not in exclude_cols_list and
               df_processed[col].dtype in ['float64', 'int64', 'float32', 'int32']]

print(f"  Found {len(sensor_cols)} sensor columns (no lagging applied)")

# 5. DEFINE TARGETS - KEEP product_id IN df_processed!
Y_cls = df_processed['failure_type_encoded'].copy()  # Don't reset_index yet!
Y_reg = df_processed['rul'].copy()

# Define features
exclude_cols = [col for col in exclude_cols_list if col in df_processed.columns]
feature_cols = [col for col in df_processed.columns if col not in exclude_cols]

# IMPORTANT: Keep df_processed with original index for product_id lookup
X = df_processed[feature_cols].copy()

print(f"\n✓ PREPROCESSING COMPLETE")
print(f"  Feature matrix (X): {X.shape}")
print(f"  Classification target (Y_cls): {Y_cls.shape}")
print(f"  Regression target (Y_reg): {Y_reg.shape}")
print(f"  df_processed retained with product_id for sequence creation")



PREPROCESSING: ENCODING & FEATURE ENGINEERING

🔍 INITIAL DATASET:
  Shape: (57912, 9)
  Columns: ['engine_type', 'air_temp', 'process_temp', 'rpm', 'torque_nm', 'tool_wear', 'failure_type', 'product_id', 'timestamp']
  Initial NaN count: 0
  Dropped 0 rows with NaN from original data
  Remaining rows: 57912

✓ Label encoders saved
  Engine types: ['H' 'L' 'M']
  Failure types: ['Heat Dissipation Failure' 'No Failure' 'Overstrain Failure'
 'Power Failure' 'Random Failures' 'Tool Wear Failure']

⚠️  Creating proper product IDs based on sequential grouping...
  ✓ Created 580 unique product IDs
  Timesteps per product:
    Mean: 99.8
    Min: 12
    Max: 100
    Products with ≥10 timesteps: 580 / 580

  Sample product_id distribution:
    First 30 rows (every 10th): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

### Creating RUL (Remaining Useful Life) ###
⚠️  'rul' column not found - Creating synthetic RUL based on failure_type...

✓ Label enc

In [24]:
# Diagnostic cell - run this to check CSV
df_check = pd.read_csv('preprocessed_data\\predictive_maintenance_synthesized.csv')
print(f"Original CSV shape: {df_check.shape}")
print(f"Total NaN values: {df_check.isnull().sum().sum()}")
print(f"\nNaN per column:")
print(df_check.isnull().sum())
print(f"\nFirst few rows:")
print(df_check.head())

Original CSV shape: (57912, 9)
Total NaN values: 0

NaN per column:
engine_type     0
air_temp        0
process_temp    0
rpm             0
torque_nm       0
tool_wear       0
failure_type    0
product_id      0
timestamp       0
dtype: int64

First few rows:
  engine_type  air_temp  process_temp       rpm  torque_nm  tool_wear  \
0           H -0.802401     -0.745157  1.356707  -1.613785  -1.318927   
1           H -0.702409     -0.610355 -0.645801   0.432675  -1.240374   
2           H -0.602417     -0.542955 -1.298429   1.455905  -0.910450   
3           H -0.552421     -0.475554 -0.913546   0.272168  -0.784764   
4           H -0.452429     -0.408153  1.518470  -1.543563  -0.486261   

  failure_type product_id            timestamp  
0   No Failure     H29424  2024-01-11 16:00:00  
1   No Failure     H29425  2024-03-20 11:00:00  
2   No Failure     H29432  2024-02-22 17:00:00  
3   No Failure     H29434  2024-01-11 15:00:00  
4   No Failure     H29441  2024-01-25 07:00:00  
Total N

### Train-Test Split and Normalization

In [25]:
print("\n" + "="*70)
print("TRAIN-TEST SPLIT & NORMALIZATION")
print("="*70)

# IMPORTANT: Don't stratify if it causes issues with product_id grouping
# For time-series, we want to keep products intact
X_train, X_test, Y_cls_train, Y_cls_test, Y_reg_train, Y_reg_test = train_test_split(
    X, Y_cls, Y_reg, 
    test_size=0.2, 
    random_state=RANDOM_SEED,
    shuffle=True  # Shuffle but maintain index
)

print(f"✓ Train-test split (80/20)")
print(f"  Training samples: {X_train.shape[0]}")
print(f"  Testing samples: {X_test.shape[0]}")
print(f"  X_train index preserved: {X_train.index[:5].tolist()}")
print(f"  X_test index preserved: {X_test.index[:5].tolist()}")

# Check class distribution in train/test
print(f"\n  Train class distribution:")
print(Y_cls_train.value_counts(normalize=True).mul(100).round(2).to_frame('percent'))
print(f"\n  Test class distribution:")
print(Y_cls_test.value_counts(normalize=True).mul(100).round(2).to_frame('percent'))

# Normalization with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for future inference
joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, 'scaler.joblib'))
print("\n✓ StandardScaler fitted and saved")
print(f"  Scaler mean: {scaler.mean_[:3]}... (first 3 features)")
print(f"  Scaler std: {scaler.scale_[:3]}... (first 3 features)")

# Convert back to DataFrame for easier handling
X_train_scaled_df = pd.DataFrame(
    X_train_scaled, 
    columns=feature_cols, 
    index=X_train.index
)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled, 
    columns=feature_cols, 
    index=X_test.index
)

print(f"✓ Scaled data converted to DataFrames")
print(f"  X_train_scaled_df: {X_train_scaled_df.shape}")
print(f"  X_test_scaled_df: {X_test_scaled_df.shape}")

# Verify no data leakage
assert len(set(X_train.index) & set(X_test.index)) == 0, "❌ Data leakage detected!"
print("\n✅ No data leakage - train and test sets are independent")


TRAIN-TEST SPLIT & NORMALIZATION
✓ Train-test split (80/20)
  Training samples: 46329
  Testing samples: 11583
  X_train index preserved: [4845, 45796, 41211, 12409, 28153]
  X_test index preserved: [16971, 53134, 46497, 9584, 1566]

  Train class distribution:
                      percent
failure_type_encoded         
0                       16.74
2                       16.73
1                       16.64
5                       16.64
3                       16.63
4                       16.63

  Test class distribution:
                      percent
failure_type_encoded         
4                       16.83
3                       16.81
5                       16.79
1                       16.76
2                       16.43
0                       16.39

✓ StandardScaler fitted and saved
  Scaler mean: [ 0.28526064  0.18061067 -0.16972473]... (first 3 features)
  Scaler std: [0.96660699 0.91792434 1.72372961]... (first 3 features)
✓ Scaled data converted to DataFrames
  X_train_

### Sequence Creation for LSTM & SMOTE-NC Balancing

In [26]:
print("\n" + "="*70)
print("DIAGNOSTIC: Product ID Distribution Analysis")
print("="*70)

# Check original df_processed
print(f"\n📊 ORIGINAL df_processed:")
print(f"  Total rows: {len(df_processed)}")
print(f"  Index range: {df_processed.index.min()} to {df_processed.index.max()}")
print(f"  Has product_id? {'product_id' in df_processed.columns}")

if 'product_id' in df_processed.columns:
    print(f"  Unique product_ids: {df_processed['product_id'].nunique()}")
    product_counts = df_processed.groupby('product_id').size()
    print(f"\n  Timesteps per product:")
    print(f"    Mean: {product_counts.mean():.2f}")
    print(f"    Median: {product_counts.median():.0f}")
    print(f"    Min: {product_counts.min()}")
    print(f"    Max: {product_counts.max()}")
    print(f"    Products with ≥10 timesteps: {(product_counts >= 10).sum()}")
    
    print(f"\n  Sample product_id distribution (first 20):")
    print(df_processed[['product_id']].head(20))

# Check X (after reset_index)
print(f"\n📊 FEATURE MATRIX X (after reset_index):")
print(f"  Shape: {X.shape}")
print(f"  Index range: {X.index.min()} to {X.index.max()}")

# Check train/test indices
print(f"\n📊 TRAIN/TEST SPLIT INDICES:")
print(f"  X_train index range: {X_train.index.min()} to {X_train.index.max()}")
print(f"  X_test index range: {X_test.index.min()} to {X_test.index.max()}")
print(f"  X_train first 10 indices: {X_train.index[:10].tolist()}")

# Try to extract product_ids
print(f"\n📊 PRODUCT_ID EXTRACTION ATTEMPT:")
try:
    test_extract = df_processed.loc[X_train.index[:10], 'product_id']
    print(f"  ✗ EXTRACTION FAILED - Index mismatch!")
    print(f"  Requested indices: {X_train.index[:10].tolist()}")
    print(f"  df_processed available indices: {df_processed.index[:10].tolist()}")
except KeyError as e:
    print(f"  ✗ KeyError: {e}")


DIAGNOSTIC: Product ID Distribution Analysis

📊 ORIGINAL df_processed:
  Total rows: 57912
  Index range: 0 to 57911
  Has product_id? True
  Unique product_ids: 580

  Timesteps per product:
    Mean: 99.85
    Median: 100
    Min: 12
    Max: 100
    Products with ≥10 timesteps: 580

  Sample product_id distribution (first 20):
    product_id
0            0
1            0
2            0
3            0
4            0
5            0
6            0
7            0
8            0
9            0
10           0
11           0
12           0
13           0
14           0
15           0
16           0
17           0
18           0
19           0

📊 FEATURE MATRIX X (after reset_index):
  Shape: (57912, 5)
  Index range: 0 to 57911

📊 TRAIN/TEST SPLIT INDICES:
  X_train index range: 0 to 57911
  X_test index range: 4 to 57899
  X_train first 10 indices: [4845, 45796, 41211, 12409, 28153, 15603, 25943, 24508, 39612, 51172]

📊 PRODUCT_ID EXTRACTION ATTEMPT:
  ✗ EXTRACTION FAILED - Index mismatc

In [27]:
print("\n" + "="*70)
print("PREPROCESSING: LSTM SEQUENCES & SMOTE-NC BALANCING")
print("="*70)

# 1. SEQUENCE CREATION FOR LSTM
def create_sequences(X_scaled_df, Y_cls, Y_reg, product_ids_series, sequence_length=10):
    """
    Create fixed-length time window sequences for LSTM
    """
    import numpy as np
    X_seq, Y_cls_seq, Y_reg_seq = [], [], []
    
    # Debug: Check alignment
    print(f"\n  DEBUG - create_sequences:")
    print(f"    X_scaled_df index range: {X_scaled_df.index.min()} to {X_scaled_df.index.max()}")
    print(f"    product_ids_series index range: {product_ids_series.index.min()} to {product_ids_series.index.max()}")
    print(f"    Unique products: {product_ids_series.nunique()}")
    
    # Ensure all indices are aligned
    if not X_scaled_df.index.equals(product_ids_series.index):
        print(f"    ⚠️  WARNING: Index mismatch - attempting to align...")
        common_idx = X_scaled_df.index.intersection(product_ids_series.index)
        X_scaled_df = X_scaled_df.loc[common_idx]
        Y_cls = Y_cls.loc[common_idx]
        Y_reg = Y_reg.loc[common_idx]
        product_ids_series = product_ids_series.loc[common_idx]
        print(f"    ✓ Aligned to {len(common_idx)} common indices")
    
    # Iterate through each product
    products_processed = 0
    sequences_created = 0
    
    for pid in product_ids_series.unique():
        # Get data for this product
        mask = product_ids_series == pid
        indices = product_ids_series[mask].index
        
        # Extract values for this product (sorted by index to maintain temporal order)
        indices_sorted = sorted(indices)
        X_prod = X_scaled_df.loc[indices_sorted].values
        Y_cls_prod = Y_cls.loc[indices_sorted].values
        Y_reg_prod = Y_reg.loc[indices_sorted].values
        
        # Create sequences if product has enough data
        if len(X_prod) >= sequence_length:
            for i in range(len(X_prod) - sequence_length + 1):
                X_seq.append(X_prod[i:i+sequence_length])
                Y_cls_seq.append(Y_cls_prod[i+sequence_length-1])
                Y_reg_seq.append(Y_reg_prod[i+sequence_length-1])
                sequences_created += 1
            products_processed += 1
    
    print(f"    ✓ Processed {products_processed} products")
    print(f"    ✓ Created {sequences_created} sequences")
    
    if sequences_created == 0:
        print(f"    ❌ ERROR: No sequences created!")
        print(f"    Possible causes:")
        print(f"      - All products have < {sequence_length} timesteps")
        print(f"      - Index alignment issue")
        return np.array([]), np.array([]), np.array([])
    
    return (np.array(X_seq), 
            np.array(Y_cls_seq), 
            np.array(Y_reg_seq))

SEQUENCE_LENGTH = 10

# Get product IDs for train and test sets
print(f"\nPreparing product IDs for sequence creation...")
print(f"  df_processed shape: {df_processed.shape}")
print(f"  df_processed index range: [{df_processed.index.min()}, {df_processed.index.max()}]")
print(f"  X_train index range: [{X_train.index.min()}, {X_train.index.max()}]")
print(f"  X_test index range: [{X_test.index.min()}, {X_test.index.max()}]")

# Extract product IDs using preserved indices
train_product_ids = df_processed.loc[X_train.index, 'product_id']
test_product_ids = df_processed.loc[X_test.index, 'product_id']

print(f"\n✓ Product IDs extracted successfully:")
print(f"  Train: {len(train_product_ids)} samples, {train_product_ids.nunique()} unique products")
print(f"  Test:  {len(test_product_ids)} samples, {test_product_ids.nunique()} unique products")

# Verify realistic distribution
train_product_counts = train_product_ids.value_counts()
print(f"\n  Train products with ≥10 timesteps: {(train_product_counts >= 10).sum()} / {train_product_ids.nunique()}")
print(f"  Average timesteps per product: {train_product_counts.mean():.1f}")

# Create sequences for train and test
X_seq_train, Y_cls_seq_train, Y_reg_seq_train = create_sequences(
    X_train_scaled_df, Y_cls_train, Y_reg_train, train_product_ids, SEQUENCE_LENGTH
)
X_seq_test, Y_cls_seq_test, Y_reg_seq_test = create_sequences(
    X_test_scaled_df, Y_cls_test, Y_reg_test, test_product_ids, SEQUENCE_LENGTH
)

# Check if sequences were created
if len(X_seq_train) == 0 or len(X_seq_test) == 0:
    raise ValueError(
        "\n❌ ERROR: Failed to create LSTM sequences!\n"
        f"Train sequences: {len(X_seq_train)}, Test sequences: {len(X_seq_test)}"
    )

print(f"\n✓ LSTM sequences created successfully")
print(f"  Train sequences: {X_seq_train.shape}")
print(f"  Test sequences: {X_seq_test.shape}")
print(f"  Features per timestep: {X_seq_train.shape[2]}")

# 2. SMOTE-NC BALANCING (Classification Only, on Training Set)
print(f"\n{'='*70}")
print("SMOTE-NC BALANCING FOR CLASSIFICATION")
print(f"{'='*70}")

# Check if engine_type_encoded is in feature_cols
if 'engine_type_encoded' in feature_cols:
    categorical_features_idx = [feature_cols.index('engine_type_encoded')]
    print(f"  ✓ Categorical feature 'engine_type_encoded' found at index {categorical_features_idx[0]}")
else:
    print(f"  ⚠️  'engine_type_encoded' not in feature columns")
    print(f"  Available features: {feature_cols}")
    print(f"  Using SMOTE instead of SMOTE-NC (all features treated as continuous)")
    categorical_features_idx = None

print(f"\n  Original class distribution:")
class_dist_orig = pd.Series(Y_cls_train).value_counts().sort_index()
for idx, count in class_dist_orig.items():
    print(f"    Class {idx} ({le_failure.classes_[idx]}): {count} ({count/len(Y_cls_train)*100:.1f}%)")

# Apply SMOTE or SMOTE-NC based on categorical feature availability
if categorical_features_idx is not None:
    from imblearn.over_sampling import SMOTENC
    print(f"\n  Applying SMOTE-NC balancing (with categorical features)...")
    smote = SMOTENC(
        categorical_features=categorical_features_idx,
        random_state=RANDOM_SEED,
        k_neighbors=5
    )
else:
    from imblearn.over_sampling import SMOTE
    print(f"\n  Applying SMOTE balancing (continuous features only)...")
    smote = SMOTE(
        random_state=RANDOM_SEED,
        k_neighbors=5
    )

X_train_balanced, Y_cls_train_balanced = smote.fit_resample(
    X_train_scaled, Y_cls_train
)

print(f"  Balanced class distribution:")
class_dist_balanced = pd.Series(Y_cls_train_balanced).value_counts().sort_index()
for idx, count in class_dist_balanced.items():
    print(f"    Class {idx} ({le_failure.classes_[idx]}): {count} ({count/len(Y_cls_train_balanced)*100:.1f}%)")

print(f"\n✓ {'SMOTE-NC' if categorical_features_idx else 'SMOTE'} balanced training set: {X_train_balanced.shape}")
print(f"  Original samples: {len(Y_cls_train)}")
print(f"  Balanced samples: {len(Y_cls_train_balanced)}")
print(f"  New samples created: {len(Y_cls_train_balanced) - len(Y_cls_train)}")

# Create balanced regression target (aligned with balanced classification)
print(f"\n  Creating aligned regression targets for balanced data...")
class_rul_means = Y_reg_train.groupby(Y_cls_train).mean()

Y_reg_train_balanced = np.zeros(len(Y_cls_train_balanced))
for i, cls_label in enumerate(Y_cls_train_balanced):
    if i < len(Y_cls_train):
        # Original sample - use original RUL
        Y_reg_train_balanced[i] = Y_reg_train.iloc[i]
    else:
        # Synthetic sample - use class mean RUL
        Y_reg_train_balanced[i] = class_rul_means[cls_label]

print(f"✓ Regression targets aligned with balanced classification targets")

# Note on LSTM sequences
print(f"\n{'='*70}")
print("NOTE: LSTM Sequence Handling")
print(f"{'='*70}")
print("  For LSTM models, using ORIGINAL (unbalanced) sequences")
print("  Reason: SMOTE/SMOTE-NC breaks temporal continuity")
print("  Alternative: Use class_weight parameter in LSTM training")
print(f"  Original sequence shapes:")
print(f"    Train: {X_seq_train.shape}")
print(f"    Test:  {X_seq_test.shape}")

# Calculate class weights for LSTM
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(Y_cls_seq_train),
    y=Y_cls_seq_train
)
class_weights_dict = dict(enumerate(class_weights_array))

print(f"\n  Computed class weights for LSTM:")
for idx, weight in class_weights_dict.items():
    print(f"    Class {idx} ({le_failure.classes_[idx]}): {weight:.3f}")

print("\n" + "="*70)
print("PREPROCESSING COMPLETE - READY FOR MODELING")
print("="*70)
print("\nDatasets prepared:")
print(f"  ✓ XGBoost (balanced):  X_train_balanced ({X_train_balanced.shape})")
print(f"  ✓ XGBoost (original):  X_test_scaled ({X_test_scaled.shape})")
print(f"  ✓ LSTM (unbalanced):   X_seq_train ({X_seq_train.shape})")
print(f"  ✓ LSTM (unbalanced):   X_seq_test ({X_seq_test.shape})")



PREPROCESSING: LSTM SEQUENCES & SMOTE-NC BALANCING

Preparing product IDs for sequence creation...
  df_processed shape: (57912, 12)
  df_processed index range: [0, 57911]


  X_train index range: [0, 57911]
  X_test index range: [4, 57899]

✓ Product IDs extracted successfully:
  Train: 46329 samples, 580 unique products
  Test:  11583 samples, 579 unique products

  Train products with ≥10 timesteps: 580 / 580
  Average timesteps per product: 79.9

  DEBUG - create_sequences:
    X_scaled_df index range: 0 to 57911
    product_ids_series index range: 0 to 57911
    Unique products: 580
    ✓ Processed 580 products
    ✓ Created 41109 sequences

  DEBUG - create_sequences:
    X_scaled_df index range: 4 to 57899
    product_ids_series index range: 4 to 57899
    Unique products: 579
    ✓ Processed 580 products
    ✓ Created 41109 sequences

  DEBUG - create_sequences:
    X_scaled_df index range: 4 to 57899
    product_ids_series index range: 4 to 57899
    Unique products: 579
    ✓ Processed 578 products
    ✓ Created 6372 sequences

✓ LSTM sequences created successfully
  Train sequences: (41109, 10, 5)
  Test sequences: (6372, 10, 5)
  Features per t

### XGBoost Classification (Anomaly Detection)

In [28]:
print("\n" + "="*70)
print("MODEL 1: XGBoost Classification (Anomaly Detection)")
print("="*70)

# Train XGBoost Classifier on SMOTE balanced data
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_SEED,
    eval_metric='mlogloss'
)

xgb_clf.fit(X_train_balanced, Y_cls_train_balanced)
print("✓ XGBoost Classifier trained")

# Predict on original test set (not SMOTE balanced)
y_pred_xgb_clf = xgb_clf.predict(X_test_scaled)
y_pred_proba_xgb_clf = xgb_clf.predict_proba(X_test_scaled)

# Evaluate metrics
acc_xgb_clf = accuracy_score(Y_cls_test, y_pred_xgb_clf)
prec_xgb_clf = precision_score(Y_cls_test, y_pred_xgb_clf, average='weighted')
rec_xgb_clf = recall_score(Y_cls_test, y_pred_xgb_clf, average='weighted')
f1_xgb_clf = f1_score(Y_cls_test, y_pred_xgb_clf, average='weighted')
roc_auc_xgb_clf = roc_auc_score(Y_cls_test, y_pred_proba_xgb_clf, 
                                 multi_class='ovr', average='weighted')
pr_auc_xgb_clf = average_precision_score(Y_cls_test, y_pred_proba_xgb_clf, 
                                          average='weighted')

print(f"\n📊 XGBoost Classification Metrics:")
print(f"  Accuracy:  {acc_xgb_clf:.4f}")
print(f"  Precision: {prec_xgb_clf:.4f}")
print(f"  Recall:    {rec_xgb_clf:.4f}")
print(f"  F1-Score:  {f1_xgb_clf:.4f}")
print(f"  ROC-AUC:   {roc_auc_xgb_clf:.4f}")
print(f"  PR-AUC:    {pr_auc_xgb_clf:.4f}")

# Save metrics
with open(os.path.join(ARTIFACTS_DIR, 'xgb_classification_metrics.txt'), 'w') as f:
    f.write("XGBoost Classification Metrics\n")
    f.write("="*50 + "\n")
    f.write(f"Accuracy:  {acc_xgb_clf:.4f}\n")
    f.write(f"Precision: {prec_xgb_clf:.4f}\n")
    f.write(f"Recall:    {rec_xgb_clf:.4f}\n")
    f.write(f"F1-Score:  {f1_xgb_clf:.4f}\n")
    f.write(f"ROC-AUC:   {roc_auc_xgb_clf:.4f}\n")
    f.write(f"PR-AUC:    {pr_auc_xgb_clf:.4f}\n")

# Confusion Matrix
cm_xgb_clf = confusion_matrix(Y_cls_test, y_pred_xgb_clf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_clf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_failure.classes_,
            yticklabels=le_failure.classes_)
plt.title('XGBoost Classification - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'xgb_classification_confusion_matrix.png'), dpi=300)
plt.close()

# ROC Curve (One-vs-Rest for multi-class)
plt.figure(figsize=(10, 6))
for i in range(len(le_failure.classes_)):
    fpr, tpr, _ = roc_curve((Y_cls_test == i).astype(int), y_pred_proba_xgb_clf[:, i])
    plt.plot(fpr, tpr, label=f'{le_failure.classes_[i]} (AUC={roc_auc_score((Y_cls_test == i).astype(int), y_pred_proba_xgb_clf[:, i]):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGBoost Classification - ROC Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'xgb_classification_roc_curves.png'), dpi=300)
plt.close()

# Save XGBoost model
joblib.dump(xgb_clf, os.path.join(ARTIFACTS_DIR, 'xgboost_classifier.pkl'))
print("✓ Model and visualizations saved")



MODEL 1: XGBoost Classification (Anomaly Detection)
✓ XGBoost Classifier trained
✓ XGBoost Classifier trained

📊 XGBoost Classification Metrics:
  Accuracy:  0.9896
  Precision: 0.9898
  Recall:    0.9896
  F1-Score:  0.9896
  ROC-AUC:   0.9995
  PR-AUC:    0.9986

📊 XGBoost Classification Metrics:
  Accuracy:  0.9896
  Precision: 0.9898
  Recall:    0.9896
  F1-Score:  0.9896
  ROC-AUC:   0.9995
  PR-AUC:    0.9986
✓ Model and visualizations saved
✓ Model and visualizations saved


### Model 2 LSTM

In [29]:
print("\n" + "="*70)
print("MODEL 2: LSTM Classification (Time-Series Anomaly Detection)")
print("="*70)

# Use sequences created earlier
print(f"Input data:")
print(f"  X_seq_train: {X_seq_train.shape}")  # (n_sequences, 10, 6)
print(f"  Y_cls_seq_train: {Y_cls_seq_train.shape}")  # (n_sequences,)
print(f"  X_seq_test: {X_seq_test.shape}")
print(f"  Y_cls_seq_test: {Y_cls_seq_test.shape}")

n_features = X_seq_train.shape[2]  # 6 features
n_classes = len(np.unique(Y_cls_seq_train))  # Number of failure types

print(f"\n🔧 Model Configuration:")
print(f"  Sequence length: {SEQUENCE_LENGTH}")
print(f"  Number of features: {n_features}")
print(f"  Number of classes: {n_classes}")

# Build LSTM model
model_lstm_clf = Sequential([
    keras.layers.Input(shape=(SEQUENCE_LENGTH, n_features)),
    keras.layers.Masking(mask_value=0.0),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(n_classes, activation='softmax')
])

model_lstm_clf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ LSTM model architecture:")
model_lstm_clf.summary()

# Train with class weights to handle imbalance
print(f"\n🏋️ Training LSTM Classifier...")
history_lstm_clf = model_lstm_clf.fit(
    X_seq_train, Y_cls_seq_train,
    validation_split=0.2,
    epochs=20,
    batch_size=256,
    class_weight=class_weights_dict,  # Use computed class weights
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ],
    verbose=1
)

print("✓ LSTM Classifier trained")

# Predict
y_pred_proba_lstm_clf = model_lstm_clf.predict(X_seq_test, verbose=0)
y_pred_lstm_clf = y_pred_proba_lstm_clf.argmax(axis=1)

# Evaluate
acc_lstm_clf = accuracy_score(Y_cls_seq_test, y_pred_lstm_clf)
prec_lstm_clf = precision_score(Y_cls_seq_test, y_pred_lstm_clf, average='weighted')
rec_lstm_clf = recall_score(Y_cls_seq_test, y_pred_lstm_clf, average='weighted')
f1_lstm_clf = f1_score(Y_cls_seq_test, y_pred_lstm_clf, average='weighted')

# ROC-AUC (One-vs-Rest for multiclass)
from sklearn.preprocessing import label_binarize
y_test_bin = label_binarize(Y_cls_seq_test, classes=np.arange(n_classes))
roc_auc_lstm_clf = roc_auc_score(y_test_bin, y_pred_proba_lstm_clf, average='weighted', multi_class='ovr')
pr_auc_lstm_clf = average_precision_score(y_test_bin, y_pred_proba_lstm_clf, average='weighted')

print(f"\n📊 LSTM Classification Metrics:")
print(f"  Accuracy:  {acc_lstm_clf:.4f}")
print(f"  Precision: {prec_lstm_clf:.4f}")
print(f"  Recall:    {rec_lstm_clf:.4f}")
print(f"  F1-Score:  {f1_lstm_clf:.4f}")
print(f"  ROC-AUC:   {roc_auc_lstm_clf:.4f}")
print(f"  PR-AUC:    {pr_auc_lstm_clf:.4f}")

# Save model
model_lstm_clf.save(os.path.join(ARTIFACTS_DIR, 'lstm_classifier.h5'))
print("✓ Model saved")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lstm_clf.history['loss'], label='Train Loss')
plt.plot(history_lstm_clf.history['val_loss'], label='Val Loss')
plt.title('LSTM Classification - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_lstm_clf.history['accuracy'], label='Train Accuracy')
plt.plot(history_lstm_clf.history['val_accuracy'], label='Val Accuracy')
plt.title('LSTM Classification - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'lstm_classification_training_history.png'), dpi=300)
plt.close()

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm_lstm_clf = confusion_matrix(Y_cls_seq_test, y_pred_lstm_clf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lstm_clf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_failure.classes_,
            yticklabels=le_failure.classes_)
plt.title('LSTM Classification - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'lstm_classification_confusion_matrix.png'), dpi=300)
plt.close()

print("✓ Visualizations saved")


MODEL 2: LSTM Classification (Time-Series Anomaly Detection)
Input data:
  X_seq_train: (41109, 10, 5)
  Y_cls_seq_train: (41109,)
  X_seq_test: (6372, 10, 5)
  Y_cls_seq_test: (6372,)

🔧 Model Configuration:
  Sequence length: 10
  Number of features: 5
  Number of classes: 6

✓ LSTM model architecture:

✓ LSTM model architecture:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_2 (Masking)             │ (None, 10, 5)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        68,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,894 (335.52 KB)

 Trainable params: 85,894 (335.52 KB)

 Non-trainable params: 0 (0.00 B)


🏋️ Training LSTM Classifier...
Epoch 1/20
Epoch 1/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.8861 - loss: 0.3763 - val_accuracy: 0.9662 - val_loss: 0.1282
Epoch 2/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.8861 - loss: 0.3763 - val_accuracy: 0.9662 - val_loss: 0.1282
Epoch 2/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9820 - loss: 0.0736 - val_accuracy: 0.9801 - val_loss: 0.0871
Epoch 3/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9820 - loss: 0.0736 - val_accuracy: 0.9801 - val_loss: 0.0871
Epoch 3/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9897 - loss: 0.0451 - val_accuracy: 0.9844 - val_loss: 0.0732
Epoch 4/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9897 - loss: 0.0451 - val_accuracy: 0.9844 - val_loss: 0.0732
Epoch 4/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9920 - loss: 0.0382 - val_accuracy: 0.9839 - val_loss: 0.0736
Epoch 5/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step -


📊 LSTM Classification Metrics:
  Accuracy:  0.9892
  Precision: 0.9893
  Recall:    0.9892
  F1-Score:  0.9892
  ROC-AUC:   0.9987
  PR-AUC:    0.9964
✓ Model saved
✓ Visualizations saved
✓ Visualizations saved


### Performance Comparison Summary

In [30]:
print("\n" + "="*70)
print("PERFORMANCE COMPARISON - CLASSIFICATION MODELS")
print("="*70)

# Create comparison dataframe for Classification models only
comparison_data = {
    'Model': [
        'XGBoost Classification',
        'LSTM Classification'
    ],
    'Type': ['Classification', 'Classification'],
    'Accuracy': [
        f'{acc_xgb_clf:.4f}',
        f'{acc_lstm_clf:.4f}'
    ],
    'Precision': [
        f'{prec_xgb_clf:.4f}',
        f'{prec_lstm_clf:.4f}'
    ],
    'Recall': [
        f'{rec_xgb_clf:.4f}',
        f'{rec_lstm_clf:.4f}'
    ],
    'F1-Score': [
        f'{f1_xgb_clf:.4f}',
        f'{f1_lstm_clf:.4f}'
    ],
    'ROC-AUC': [
        f'{roc_auc_xgb_clf:.4f}',
        f'{roc_auc_lstm_clf:.4f}'
    ],
    'PR-AUC': [
        f'{pr_auc_xgb_clf:.4f}',
        f'{pr_auc_lstm_clf:.4f}'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

# Save comparison table
comparison_df.to_csv(os.path.join(ARTIFACTS_DIR, 'classification_model_comparison.csv'), index=False)
print(f"\n✓ Classification model comparison saved")

# Note about regression models
print("\n" + "="*70)
print("NOTE: Regression Model Comparison")
print("="*70)
print("  Regression models (XGBoost & LSTM) will be compared after training")
print("  Full comparison table will be generated after all 4 models are trained")



PERFORMANCE COMPARISON - CLASSIFICATION MODELS


                 Model           Type Accuracy Precision Recall F1-Score ROC-AUC PR-AUC
XGBoost Classification Classification   0.9896    0.9898 0.9896   0.9896  0.9995 0.9986
   LSTM Classification Classification   0.9892    0.9893 0.9892   0.9892  0.9987 0.9964

✓ Classification model comparison saved

NOTE: Regression Model Comparison
  Regression models (XGBoost & LSTM) will be compared after training
  Full comparison table will be generated after all 4 models are trained


### Success Indicator Validation

In [31]:
print("\n" + "="*70)
print("SUCCESS INDICATOR VALIDATION (>70% Recall/Accuracy)")
print("="*70)

print("\n📊 Classification Models Performance:")
print(f"  XGBoost - Accuracy: {acc_xgb_clf:.2%} | Recall: {rec_xgb_clf:.2%}")
print(f"  LSTM    - Accuracy: {acc_lstm_clf:.2%} | Recall: {rec_lstm_clf:.2%}")

# Check success criteria
xgb_clf_success = (acc_xgb_clf > 0.70) and (rec_xgb_clf > 0.70)
lstm_clf_success = (acc_lstm_clf > 0.70) and (rec_lstm_clf > 0.70)

print("\n✅ SUCCESS CRITERIA (>70%):")
if xgb_clf_success:
    print(f"  ✓ XGBoost Classification MEETS the target (Acc: {acc_xgb_clf:.2%}, Rec: {rec_xgb_clf:.2%})")
else:
    print(f"  ✗ XGBoost Classification BELOW target (Acc: {acc_xgb_clf:.2%}, Rec: {rec_xgb_clf:.2%})")

if lstm_clf_success:
    print(f"  ✓ LSTM Classification MEETS the target (Acc: {acc_lstm_clf:.2%}, Rec: {rec_lstm_clf:.2%})")
else:
    print(f"  ✗ LSTM Classification BELOW target (Acc: {acc_lstm_clf:.2%}, Rec: {rec_lstm_clf:.2%})")

# Determine best classifier
best_classifier = 'XGBoost' if acc_xgb_clf >= acc_lstm_clf else 'LSTM'

print(f"\n🏆 BEST PERFORMING CLASSIFIER:")
print(f"  Best Classifier: {best_classifier}")
print(f"\n📝 NOTE: Regression models will be evaluated after training")



SUCCESS INDICATOR VALIDATION (>70% Recall/Accuracy)

📊 Classification Models Performance:
  XGBoost - Accuracy: 98.96% | Recall: 98.96%
  LSTM    - Accuracy: 98.92% | Recall: 98.92%

✅ SUCCESS CRITERIA (>70%):
  ✓ XGBoost Classification MEETS the target (Acc: 98.96%, Rec: 98.96%)
  ✓ LSTM Classification MEETS the target (Acc: 98.92%, Rec: 98.92%)

🏆 BEST PERFORMING CLASSIFIER:
  Best Classifier: XGBoost

📝 NOTE: Regression models will be evaluated after training


### XGBoost Regression (RUL Prediction)

In [32]:
print("\n" + "="*70)
print("MODEL 3: XGBoost Regression (RUL Prediction)")
print("="*70)

# Augment features with classification probabilities for better RUL prediction
X_train_aug = np.concatenate([X_train_balanced, 
                                xgb_clf.predict_proba(X_train_balanced)], axis=1)
X_test_aug = np.concatenate([X_test_scaled, 
                              xgb_clf.predict_proba(X_test_scaled)], axis=1)

print(f"Augmented features shape:")
print(f"  Train: {X_train_aug.shape}")
print(f"  Test:  {X_test_aug.shape}")

# Train XGBoost Regressor
xgb_reg = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_SEED,
    eval_metric='rmse'
)

xgb_reg.fit(X_train_aug, Y_reg_train_balanced)
print("✓ XGBoost Regressor trained")

# Predict
y_pred_xgb_reg = xgb_reg.predict(X_test_aug)

# Evaluate
mse_xgb = mean_squared_error(Y_reg_test, y_pred_xgb_reg)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(Y_reg_test, y_pred_xgb_reg)

print(f"\n📊 XGBoost Regression Metrics:")
print(f"  RMSE: {rmse_xgb:.4f} days")
print(f"  MAE:  {mae_xgb:.4f} days")
print(f"  MSE:  {mse_xgb:.4f}")

# Save metrics
with open(os.path.join(ARTIFACTS_DIR, 'xgb_regression_metrics.txt'), 'w') as f:
    f.write("XGBoost Regression Metrics\n")
    f.write("="*50 + "\n")
    f.write(f"RMSE: {rmse_xgb:.4f} days\n")
    f.write(f"MAE:  {mae_xgb:.4f} days\n")
    f.write(f"MSE:  {mse_xgb:.4f}\n")

# Scatter plot: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(Y_reg_test, y_pred_xgb_reg, alpha=0.5, s=10)
plt.plot([Y_reg_test.min(), Y_reg_test.max()], 
         [Y_reg_test.min(), Y_reg_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual RUL (days)')
plt.ylabel('Predicted RUL (days)')
plt.title(f'XGBoost Regression - Actual vs Predicted RUL\nRMSE: {rmse_xgb:.2f} days')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'xgb_regression_scatter.png'), dpi=300)
plt.close()

# Residual plot
residuals_xgb = Y_reg_test - y_pred_xgb_reg
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_xgb_reg, residuals_xgb, alpha=0.5, s=10)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted RUL (days)')
plt.ylabel('Residuals (days)')
plt.title('XGBoost Regression - Residual Plot')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'xgb_regression_residuals.png'), dpi=300)
plt.close()

# Save model
joblib.dump(xgb_reg, os.path.join(ARTIFACTS_DIR, 'xgboost_regressor.pkl'))
print("✓ Model and visualizations saved")



MODEL 3: XGBoost Regression (RUL Prediction)
Augmented features shape:
  Train: (46524, 11)
  Test:  (11583, 11)
Augmented features shape:
  Train: (46524, 11)
  Test:  (11583, 11)
✓ XGBoost Regressor trained

📊 XGBoost Regression Metrics:
  RMSE: 28.6318 days
  MAE:  19.7618 days
  MSE:  819.7809
✓ XGBoost Regressor trained

📊 XGBoost Regression Metrics:
  RMSE: 28.6318 days
  MAE:  19.7618 days
  MSE:  819.7809
✓ Model and visualizations saved
✓ Model and visualizations saved


### LSTM Regression (RUL Prediction)

In [33]:
print("\n" + "="*70)
print("MODEL 4: LSTM Regression (RUL Prediction)")
print("="*70)

# Use sequences created earlier
print(f"Input data:")
print(f"  X_seq_train: {X_seq_train.shape}")
print(f"  Y_reg_seq_train: {Y_reg_seq_train.shape}")
print(f"  X_seq_test: {X_seq_test.shape}")
print(f"  Y_reg_seq_test: {Y_reg_seq_test.shape}")

n_features = X_seq_train.shape[2]

print(f"\n🔧 Model Configuration:")
print(f"  Sequence length: {SEQUENCE_LENGTH}")
print(f"  Number of features: {n_features}")

# Build LSTM Regression model
model_lstm_reg = Sequential([
    keras.layers.Input(shape=(SEQUENCE_LENGTH, n_features)),
    keras.layers.Masking(mask_value=0.0),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='linear')  # Linear activation for regression
])

model_lstm_reg.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print("\n✓ LSTM Regression model architecture:")
model_lstm_reg.summary()

# Train
print(f"\n🏋️ Training LSTM Regressor...")
history_lstm_reg = model_lstm_reg.fit(
    X_seq_train, Y_reg_seq_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ],
    verbose=1
)

print("✓ LSTM Regressor trained")

# Predict
y_pred_lstm_reg = model_lstm_reg.predict(X_seq_test, verbose=0).flatten()

# Evaluate
mse_lstm = mean_squared_error(Y_reg_seq_test, y_pred_lstm_reg)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(Y_reg_seq_test, y_pred_lstm_reg)

print(f"\n📊 LSTM Regression Metrics:")
print(f"  RMSE: {rmse_lstm:.4f} days")
print(f"  MAE:  {mae_lstm:.4f} days")
print(f"  MSE:  {mse_lstm:.4f}")

# Save metrics
with open(os.path.join(ARTIFACTS_DIR, 'lstm_regression_metrics.txt'), 'w') as f:
    f.write("LSTM Regression Metrics\n")
    f.write("="*50 + "\n")
    f.write(f"RMSE: {rmse_lstm:.4f} days\n")
    f.write(f"MAE:  {mae_lstm:.4f} days\n")
    f.write(f"MSE:  {mse_lstm:.4f}\n")

# Save model
model_lstm_reg.save(os.path.join(ARTIFACTS_DIR, 'lstm_regressor.h5'))
print("✓ Model saved")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lstm_reg.history['loss'], label='Train Loss')
plt.plot(history_lstm_reg.history['val_loss'], label='Val Loss')
plt.title('LSTM Regression - Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_lstm_reg.history['mae'], label='Train MAE')
plt.plot(history_lstm_reg.history['val_mae'], label='Val MAE')
plt.title('LSTM Regression - MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE (days)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'lstm_regression_training_history.png'), dpi=300)
plt.close()

# Scatter plot: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(Y_reg_seq_test, y_pred_lstm_reg, alpha=0.5, s=10)
plt.plot([Y_reg_seq_test.min(), Y_reg_seq_test.max()], 
         [Y_reg_seq_test.min(), Y_reg_seq_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual RUL (days)')
plt.ylabel('Predicted RUL (days)')
plt.title(f'LSTM Regression - Actual vs Predicted RUL\nRMSE: {rmse_lstm:.2f} days')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'lstm_regression_scatter.png'), dpi=300)
plt.close()

# Residual plot
residuals_lstm = Y_reg_seq_test - y_pred_lstm_reg
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_lstm_reg, residuals_lstm, alpha=0.5, s=10)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted RUL (days)')
plt.ylabel('Residuals (days)')
plt.title('LSTM Regression - Residual Plot')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'lstm_regression_residuals.png'), dpi=300)
plt.close()

print("✓ Visualizations saved")



MODEL 4: LSTM Regression (RUL Prediction)
Input data:
  X_seq_train: (41109, 10, 5)
  Y_reg_seq_train: (41109,)
  X_seq_test: (6372, 10, 5)
  Y_reg_seq_test: (6372,)

🔧 Model Configuration:
  Sequence length: 10
  Number of features: 5

✓ LSTM Regression model architecture:


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_3 (Masking)             │ (None, 10, 5)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 10, 128)        │        68,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,241 (477.50 KB)

 Trainable params: 122,241 (477.50 KB)

 Non-trainable params: 0 (0.00 B)


🏋️ Training LSTM Regressor...
Epoch 1/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - loss: 11163.3857 - mae: 73.0525 - val_loss: 7006.3330 - val_mae: 56.8507
Epoch 2/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - loss: 11163.3857 - mae: 73.0525 - val_loss: 7006.3330 - val_mae: 56.8507
Epoch 2/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 6871.2769 - mae: 61.4062 - val_loss: 6706.4058 - val_mae: 58.6631
Epoch 3/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 6871.2769 - mae: 61.4062 - val_loss: 6706.4058 - val_mae: 58.6631
Epoch 3/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 6708.9956 - mae: 60.7702 - val_loss: 6074.8276 - val_mae: 55.4345
Epoch 4/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 6708.9956 - mae: 60.7702 - val_loss: 6074.8276 - val_mae: 55.4345
Epoch 4/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 3282.5037 - mae: 36.1501 - val_loss: 1941.1000 - val_mae: 28.6990
Epoch 5/30
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 3282.5037 - mae: 


📊 LSTM Regression Metrics:
  RMSE: 31.7018 days
  MAE:  21.2648 days
  MSE:  1005.0013
✓ Model saved
✓ Visualizations saved
✓ Visualizations saved


### Final Model Comparison (All 4 Models)

In [34]:
print("\n" + "="*70)
print("FINAL MODEL COMPARISON - ALL 4 MODELS")
print("="*70)

# Classification Models Comparison
print("\n📊 CLASSIFICATION MODELS:")
print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'ROC-AUC':<12}")
print("-" * 85)
print(f"{'XGBoost Classification':<25} {acc_xgb_clf:<12.4f} {prec_xgb_clf:<12.4f} {rec_xgb_clf:<12.4f} {f1_xgb_clf:<12.4f} {roc_auc_xgb_clf:<12.4f}")
print(f"{'LSTM Classification':<25} {acc_lstm_clf:<12.4f} {prec_lstm_clf:<12.4f} {rec_lstm_clf:<12.4f} {f1_lstm_clf:<12.4f} {roc_auc_lstm_clf:<12.4f}")

# Regression Models Comparison
print("\n📊 REGRESSION MODELS:")
print(f"{'Model':<25} {'RMSE (days)':<15} {'MAE (days)':<15} {'MSE':<15}")
print("-" * 70)
print(f"{'XGBoost Regression':<25} {rmse_xgb:<15.4f} {mae_xgb:<15.4f} {mse_xgb:<15.4f}")
print(f"{'LSTM Regression':<25} {rmse_lstm:<15.4f} {mae_lstm:<15.4f} {mse_lstm:<15.4f}")

# Determine best models
best_classifier = 'XGBoost' if acc_xgb_clf >= acc_lstm_clf else 'LSTM'
best_regressor = 'XGBoost' if rmse_xgb <= rmse_lstm else 'LSTM'

print("\n" + "="*70)
print("🏆 BEST PERFORMING MODELS")
print("="*70)
print(f"  Best Classifier: {best_classifier} Classification")
if best_classifier == 'XGBoost':
    print(f"    - Accuracy: {acc_xgb_clf:.2%}, Recall: {rec_xgb_clf:.2%}")
else:
    print(f"    - Accuracy: {acc_lstm_clf:.2%}, Recall: {rec_lstm_clf:.2%}")

print(f"\n  Best Regressor: {best_regressor} Regression")
if best_regressor == 'XGBoost':
    print(f"    - RMSE: {rmse_xgb:.2f} days, MAE: {mae_xgb:.2f} days")
else:
    print(f"    - RMSE: {rmse_lstm:.2f} days, MAE: {mae_lstm:.2f} days")

# Success criteria check
print("\n" + "="*70)
print("✅ SUCCESS CRITERIA VALIDATION (>70%)")
print("="*70)
xgb_clf_success = (acc_xgb_clf > 0.70) and (rec_xgb_clf > 0.70)
lstm_clf_success = (acc_lstm_clf > 0.70) and (rec_lstm_clf > 0.70)

if xgb_clf_success:
    print(f"  ✓ XGBoost Classification EXCEEDS target (Acc: {acc_xgb_clf:.2%}, Rec: {rec_xgb_clf:.2%})")
else:
    print(f"  ✗ XGBoost Classification BELOW target (Acc: {acc_xgb_clf:.2%}, Rec: {rec_xgb_clf:.2%})")

if lstm_clf_success:
    print(f"  ✓ LSTM Classification EXCEEDS target (Acc: {acc_lstm_clf:.2%}, Rec: {rec_lstm_clf:.2%})")
else:
    print(f"  ✗ LSTM Classification BELOW target (Acc: {acc_lstm_clf:.2%}, Rec: {rec_lstm_clf:.2%})")

# Create comprehensive comparison DataFrame
full_comparison_data = {
    'Model': [
        'XGBoost Classification',
        'LSTM Classification',
        'XGBoost Regression',
        'LSTM Regression'
    ],
    'Type': ['Classification', 'Classification', 'Regression', 'Regression'],
    'Primary Metric': [
        f'Acc: {acc_xgb_clf:.4f}',
        f'Acc: {acc_lstm_clf:.4f}',
        f'RMSE: {rmse_xgb:.4f}',
        f'RMSE: {rmse_lstm:.4f}'
    ],
    'Secondary Metric': [
        f'Recall: {rec_xgb_clf:.4f}',
        f'Recall: {rec_lstm_clf:.4f}',
        f'MAE: {mae_xgb:.4f}',
        f'MAE: {mae_lstm:.4f}'
    ],
    'ROC-AUC/MSE': [
        f'{roc_auc_xgb_clf:.4f}',
        f'{roc_auc_lstm_clf:.4f}',
        f'{mse_xgb:.4f}',
        f'{mse_lstm:.4f}'
    ]
}

full_comparison_df = pd.DataFrame(full_comparison_data)
full_comparison_df.to_csv(os.path.join(ARTIFACTS_DIR, 'full_model_comparison.csv'), index=False)
print(f"\n✓ Full model comparison saved to: {ARTIFACTS_DIR}/full_model_comparison.csv")



FINAL MODEL COMPARISON - ALL 4 MODELS

📊 CLASSIFICATION MODELS:
Model                     Accuracy     Precision    Recall       F1-Score     ROC-AUC     
-------------------------------------------------------------------------------------
XGBoost Classification    0.9896       0.9898       0.9896       0.9896       0.9995      
LSTM Classification       0.9892       0.9893       0.9892       0.9892       0.9987      

📊 REGRESSION MODELS:
Model                     RMSE (days)     MAE (days)      MSE            
----------------------------------------------------------------------
XGBoost Regression        28.6318         19.7618         819.7809       
LSTM Regression           31.7018         21.2648         1005.0013      

🏆 BEST PERFORMING MODELS
  Best Classifier: XGBoost Classification
    - Accuracy: 98.96%, Recall: 98.96%

  Best Regressor: XGBoost Regression
    - RMSE: 28.63 days, MAE: 19.76 days

✅ SUCCESS CRITERIA VALIDATION (>70%)
  ✓ XGBoost Classification EXCEEDS tar

In [35]:
print("\n" + "="*70)
print("INFERENCE PIPELINE FOR MAINTENANCE RECOMMENDATIONS")
print("="*70)

def generate_maintenance_recommendations(test_data, product_ids, 
                                          classifier_model, regressor_model,
                                          classifier_type='xgboost',
                                          regressor_type='xgboost',
                                          test_data_flat=None):
    """
    Generate maintenance recommendations for test data
    
    Parameters:
    -----------
    test_data : array-like
        Preprocessed test features (for classifier)
    product_ids : array-like
        Product IDs corresponding to test data
    classifier_model : model object
        Trained classification model
    regressor_model : model object
        Trained regression model
    classifier_type : str
        Type of classifier ('xgboost' or 'lstm')
    regressor_type : str
        Type of regressor ('xgboost' or 'lstm')
    test_data_flat : array-like, optional
        Flat test data for XGBoost regressor (when classifier is LSTM)
    
    Returns:
    --------
    recommendations_df : DataFrame
        Maintenance recommendations
    """
    
    recommendations = []
    
    # Classification predictions
    if classifier_type == 'xgboost':
        failure_preds = classifier_model.predict(test_data)
        failure_probs = classifier_model.predict_proba(test_data)
    else:  # LSTM
        failure_probs = classifier_model.predict(test_data)
        failure_preds = failure_probs.argmax(axis=1)
    
    # Regression predictions (RUL)
    if regressor_type == 'xgboost':
        # For XGBoost regressor, need flat (2D) data
        if classifier_type == 'lstm' and test_data_flat is not None:
            # Use provided flat data for XGBoost regressor
            # Align failure_probs with test_data_flat length
            failure_probs_aligned = failure_probs[:len(test_data_flat)]
            test_data_aug = np.concatenate([test_data_flat, failure_probs_aligned], axis=1)
        else:
            # Both are XGBoost, concatenate directly
            test_data_aug = np.concatenate([test_data, failure_probs], axis=1)
        rul_preds = regressor_model.predict(test_data_aug)
    else:  # LSTM
        rul_preds = regressor_model.predict(test_data).flatten()
    
    # Generate recommendations for each machine
    for i, (pid, failure_class, rul) in enumerate(zip(product_ids, failure_preds, rul_preds)):
        failure_type = le_failure.inverse_transform([failure_class])[0]
        
        # Only recommend maintenance for predicted failures (not 'No Failure')
        if failure_type != 'No Failure' and failure_type != 'Normal':
            days = int(round(rul))
            recommendation = f"Machine {pid} in {days} days needs maintenance because of {failure_type}"
            
            recommendations.append({
                'product_id': pid,
                'predicted_failure_type': failure_type,
                'predicted_rul_days': days,
                'recommendation': recommendation,
                'confidence': failure_probs[i].max()
            })
    
    return pd.DataFrame(recommendations)

# Select best models based on performance
if best_classifier == 'XGBoost':
    best_clf_model = xgb_clf
    clf_type = 'xgboost'
    test_data_clf = X_test_scaled
    test_product_ids = df_processed.loc[X_test.index, 'product_id'].values
    test_data_flat = X_test_scaled  # For XGBoost regressor
else:
    best_clf_model = model_lstm_clf  # FIXED: Use correct variable name
    clf_type = 'lstm'
    test_data_clf = X_seq_test
    # Get product IDs for sequences (use last sequence element)
    test_product_ids = df_processed.loc[X_test.index, 'product_id'].values[:len(X_seq_test)]
    test_data_flat = X_test_scaled[:len(X_seq_test)]  # For XGBoost regressor compatibility

if best_regressor == 'XGBoost':
    best_reg_model = xgb_reg
    reg_type = 'xgboost'
else:
    best_reg_model = model_lstm_reg  # FIXED: Use correct variable name
    reg_type = 'lstm'

print(f"\n🔧 Generating recommendations using:")
print(f"  Classifier: {best_classifier}")
print(f"  Regressor:  {best_regressor}")

# Generate recommendations
recommendations_df = generate_maintenance_recommendations(
    test_data=test_data_clf,
    product_ids=test_product_ids,
    classifier_model=best_clf_model,
    regressor_model=best_reg_model,
    classifier_type=clf_type,
    regressor_type=reg_type,
    test_data_flat=test_data_flat  # Pass flat data for XGBoost regressor
)

print(f"\n✓ Generated {len(recommendations_df)} maintenance recommendations")
print("\nSample recommendations:")
print(recommendations_df.head(10).to_string(index=False))

# Save recommendations
recommendations_df.to_csv(
    os.path.join(ARTIFACTS_DIR, 'maintenance_recommendations.csv'),
    index=False
)
print(f"\n✓ Recommendations saved to: {ARTIFACTS_DIR}/maintenance_recommendations.csv")


INFERENCE PIPELINE FOR MAINTENANCE RECOMMENDATIONS

🔧 Generating recommendations using:
  Classifier: XGBoost
  Regressor:  XGBoost

✓ Generated 9740 maintenance recommendations

Sample recommendations:
 product_id   predicted_failure_type  predicted_rul_days                                                              recommendation  confidence
        169       Overstrain Failure                  25      Machine 169 in 25 days needs maintenance because of Overstrain Failure    0.999090
        464        Tool Wear Failure                  42       Machine 464 in 42 days needs maintenance because of Tool Wear Failure    0.987615
         95 Heat Dissipation Failure                  91 Machine 95 in 91 days needs maintenance because of Heat Dissipation Failure    0.999313
         15        Tool Wear Failure                  48        Machine 15 in 48 days needs maintenance because of Tool Wear Failure    0.900263
        545            Power Failure                  59           Mach

## 🔧 Improved Preprocessing (Recommended Implementation)

**⚠️ NOTE:** The following cells demonstrate recommended improvements. These are **NOT applied** to the current pipeline but serve as a reference for production deployment.

In [36]:
# ========================================================================
# IMPROVED RUL GENERATION (Avoiding Data Leakage)
# ========================================================================
# This approach generates RUL independently, then derives failure types

def generate_realistic_rul(df, product_id_col='product_id'):
    """
    Generate RUL based on equipment degradation curves, not failure types.
    
    Strategy:
    1. Model equipment health as degrading function
    2. Add realistic noise and variability
    3. Derive failure type from RUL thresholds (not vice versa)
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe with sensor readings
    product_id_col : str
        Name of product ID column
        
    Returns:
    --------
    df : DataFrame
        Dataframe with added 'rul' and 'derived_failure_type' columns
    """
    
    df_improved = df.copy()
    rul_values = []
    
    # Group by product for temporal continuity
    for product_id, group in df_improved.groupby(product_id_col):
        n_timesteps = len(group)
        
        # Initialize RUL based on product lifecycle position
        # Degradation curve: Exponential decay with noise
        base_lifecycle = np.random.randint(200, 400)  # Variable lifecycle
        
        for i in range(n_timesteps):
            # Calculate position in lifecycle (0 to 1)
            lifecycle_position = i / n_timesteps
            
            # Exponential degradation with noise
            degradation = np.exp(-lifecycle_position * 2)  # Faster degradation over time
            noise = np.random.normal(0, 0.1)  # ±10% noise
            
            # RUL decreases as degradation increases
            rul = base_lifecycle * degradation + noise * 50
            rul = max(1, int(rul))  # Ensure positive
            
            # Adjust based on sensor readings (but not deterministically!)
            tool_wear = group.iloc[i]['tool_wear']
            temp = group.iloc[i]['process_temp']
            
            # Higher wear/temp → slightly lower RUL (but with randomness)
            if tool_wear > group['tool_wear'].quantile(0.75):
                rul *= np.random.uniform(0.7, 0.9)  # Random reduction
            if temp > group['process_temp'].quantile(0.75):
                rul *= np.random.uniform(0.8, 0.95)  # Random reduction
                
            rul_values.append(int(rul))
    
    df_improved['rul'] = rul_values
    
    # NOW derive failure type from RUL (breaking the deterministic link)
    def classify_failure_from_rul(rul, sensor_values):
        """
        Classify failure type based on RUL AND sensor patterns
        (not purely from RUL to add variability)
        """
        tool_wear = sensor_values['tool_wear']
        temp = sensor_values['process_temp']
        rpm = sensor_values['rpm']
        
        # Use RUL as primary indicator, but add sensor-based logic
        if rul > 150:
            return 'No Failure'
        elif rul > 100:
            # Random between No Failure and minor issues
            return np.random.choice(['No Failure', 'Random Failures'], p=[0.7, 0.3])
        elif rul > 60:
            # Check sensor patterns for specific failure types
            if tool_wear > 200:
                return 'Tool Wear Failure'
            elif temp > 310:
                return 'Heat Dissipation Failure'
            else:
                return 'Random Failures'
        elif rul > 30:
            # More likely to have specific failures
            if tool_wear > 180:
                return 'Tool Wear Failure'
            elif rpm > 2500:
                return 'Overstrain Failure'
            elif temp > 305:
                return 'Heat Dissipation Failure'
            else:
                return 'Power Failure'
        else:  # rul <= 30 (critical)
            # Imminent failures
            return np.random.choice([
                'Tool Wear Failure',
                'Overstrain Failure',
                'Heat Dissipation Failure',
                'Power Failure'
            ], p=[0.4, 0.3, 0.2, 0.1])
    
    df_improved['derived_failure_type'] = df_improved.apply(
        lambda row: classify_failure_from_rul(row['rul'], row),
        axis=1
    )
    
    return df_improved

# Example usage (commented out to not modify current pipeline):
# df_improved = generate_realistic_rul(df_processed, 'product_id')

print("✓ Improved RUL generation function defined")
print("  Key improvements:")
print("    - RUL generated independently from failure types")
print("    - Exponential degradation curves with noise")
print("    - Failure type derived from RUL + sensor patterns")
print("    - Breaks deterministic RUL ↔ failure_type mapping")

✓ Improved RUL generation function defined
  Key improvements:
    - RUL generated independently from failure types
    - Exponential degradation curves with noise
    - Failure type derived from RUL + sensor patterns
    - Breaks deterministic RUL ↔ failure_type mapping


In [37]:
# ========================================================================
# TEMPORAL FEATURE ENGINEERING (Critical for Time-Series)
# ========================================================================

def add_temporal_features(df, product_id_col='product_id', lag_periods=[1, 3, 5, 10]):
    """
    Add lag features, rates of change, and rolling statistics.
    
    These features capture temporal dynamics that are CRITICAL for
    predictive maintenance:
    - How fast is equipment degrading?
    - Are sensor values trending up or down?
    - What's the recent history of sensor readings?
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe with time-series data
    product_id_col : str
        Column name for product grouping
    lag_periods : list
        List of lag periods to create
        
    Returns:
    --------
    df : DataFrame
        Dataframe with added temporal features
    """
    
    df_temporal = df.copy()
    
    # Sensor columns to engineer
    sensor_cols = ['air_temp', 'process_temp', 'rpm', 'torque_nm', 'tool_wear']
    
    print(f"\n📊 Creating temporal features for {len(sensor_cols)} sensors...")
    
    for sensor in sensor_cols:
        print(f"  Processing: {sensor}")
        
        # 1. LAG FEATURES (Past values)
        for lag in lag_periods:
            col_name = f'{sensor}_lag_{lag}'
            df_temporal[col_name] = df_temporal.groupby(product_id_col)[sensor].shift(lag)
            
        # 2. RATE OF CHANGE (Velocity of degradation)
        df_temporal[f'{sensor}_rate'] = (
            df_temporal[sensor] - 
            df_temporal.groupby(product_id_col)[sensor].shift(1)
        )
        
        # 3. ACCELERATION (Change in rate)
        df_temporal[f'{sensor}_acceleration'] = (
            df_temporal[f'{sensor}_rate'] - 
            df_temporal.groupby(product_id_col)[f'{sensor}_rate'].shift(1)
        )
        
        # 4. ROLLING STATISTICS (Recent trends)
        for window in [3, 5, 10]:
            # Rolling mean (average recent behavior)
            df_temporal[f'{sensor}_rolling_mean_{window}'] = (
                df_temporal.groupby(product_id_col)[sensor]
                .rolling(window=window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            
            # Rolling std (volatility/instability indicator)
            df_temporal[f'{sensor}_rolling_std_{window}'] = (
                df_temporal.groupby(product_id_col)[sensor]
                .rolling(window=window, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
            )
            
        # 5. MIN/MAX in recent window (extremes)
        df_temporal[f'{sensor}_rolling_min_10'] = (
            df_temporal.groupby(product_id_col)[sensor]
            .rolling(window=10, min_periods=1)
            .min()
            .reset_index(level=0, drop=True)
        )
        df_temporal[f'{sensor}_rolling_max_10'] = (
            df_temporal.groupby(product_id_col)[sensor]
            .rolling(window=10, min_periods=1)
            .max()
            .reset_index(level=0, drop=True)
        )
    
    # 6. INTERACTION FEATURES (sensor combinations)
    print(f"\n  Creating interaction features...")
    df_temporal['temp_wear_interaction'] = (
        df_temporal['process_temp'] * df_temporal['tool_wear']
    )
    df_temporal['rpm_torque_power'] = (
        df_temporal['rpm'] * df_temporal['torque_nm']
    )
    
    # Count new features created
    new_features = [col for col in df_temporal.columns if col not in df.columns]
    print(f"\n✓ Created {len(new_features)} temporal features")
    print(f"  Original features: {len(df.columns)}")
    print(f"  Total features: {len(df_temporal.columns)}")
    
    return df_temporal

# Example usage (commented out to not modify current pipeline):
# df_with_temporal = add_temporal_features(df_processed, 'product_id')

print("\n✓ Temporal feature engineering function defined")
print("  Features added per sensor:")
print("    - Lag features (1, 3, 5, 10 timesteps)")
print("    - Rate of change (velocity)")
print("    - Acceleration (change in velocity)")
print("    - Rolling mean (3, 5, 10 windows)")
print("    - Rolling std (3, 5, 10 windows)")
print("    - Rolling min/max (10 window)")
print("    - Interaction features (temperature × wear, rpm × torque)")


✓ Temporal feature engineering function defined
  Features added per sensor:
    - Lag features (1, 3, 5, 10 timesteps)
    - Rate of change (velocity)
    - Acceleration (change in velocity)
    - Rolling mean (3, 5, 10 windows)
    - Rolling std (3, 5, 10 windows)
    - Rolling min/max (10 window)
    - Interaction features (temperature × wear, rpm × torque)


In [38]:
# ========================================================================
# TEMPORAL TRAIN-TEST SPLIT (Avoiding Future Data Leakage)
# ========================================================================

def temporal_train_test_split(df, time_col='timestamp', test_size=0.2):
    """
    Split data based on time, not randomly.
    
    Why this matters:
    - Random split: Future data can appear in training set (LEAKAGE!)
    - Temporal split: Training only uses past data (REALISTIC!)
    
    In production, you train on historical data and predict future failures.
    This split mimics that scenario.
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe with time column
    time_col : str
        Name of timestamp column
    test_size : float
        Proportion of data for testing (default 0.2 = 20%)
        
    Returns:
    --------
    train_df, test_df : tuple of DataFrames
        Temporally split training and testing sets
    """
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
    
    # Sort by time
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    
    # Calculate split point
    split_idx = int(len(df_sorted) * (1 - test_size))
    split_time = df_sorted.iloc[split_idx][time_col]
    
    # Split based on time
    train_df = df_sorted[df_sorted[time_col] <= split_time].copy()
    test_df = df_sorted[df_sorted[time_col] > split_time].copy()
    
    print(f"\n📅 TEMPORAL TRAIN-TEST SPLIT")
    print(f"{'='*70}")
    print(f"  Total samples: {len(df_sorted):,}")
    print(f"  Split time: {split_time}")
    print(f"\n  Training set:")
    print(f"    Samples: {len(train_df):,} ({len(train_df)/len(df_sorted)*100:.1f}%)")
    print(f"    Time range: {train_df[time_col].min()} to {train_df[time_col].max()}")
    print(f"\n  Test set:")
    print(f"    Samples: {len(test_df):,} ({len(test_df)/len(df_sorted)*100:.1f}%)")
    print(f"    Time range: {test_df[time_col].min()} to {test_df[time_col].max()}")
    
    # Verify no temporal leakage
    assert train_df[time_col].max() <= test_df[time_col].min(), "❌ Temporal leakage detected!"
    print(f"\n  ✅ No temporal leakage: Train data entirely before test data")
    
    return train_df, test_df

# Alternative: Product-wise temporal split (even more realistic)
def temporal_train_test_split_by_product(df, product_id_col='product_id', 
                                          time_col='timestamp', 
                                          test_products_ratio=0.2):
    """
    Split by products (not timesteps) for even more realistic evaluation.
    
    Why this is better:
    - Tests model on completely new equipment (not seen in training)
    - Mimics deployment scenario: predict failures for new machines
    - Avoids learning product-specific patterns
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe
    product_id_col : str
        Product ID column
    time_col : str
        Timestamp column
    test_products_ratio : float
        Ratio of products for testing
        
    Returns:
    --------
    train_df, test_df : tuple of DataFrames
        Training and testing sets (by product)
    """
    
    # Get unique products
    products = df[product_id_col].unique()
    n_test_products = int(len(products) * test_products_ratio)
    
    # Sort products by their first appearance time
    product_first_times = df.groupby(product_id_col)[time_col].min().sort_values()
    
    # Latest products for testing (newest equipment)
    test_products = product_first_times.tail(n_test_products).index
    train_products = product_first_times.head(len(products) - n_test_products).index
    
    # Split data
    train_df = df[df[product_id_col].isin(train_products)].copy()
    test_df = df[df[product_id_col].isin(test_products)].copy()
    
    print(f"\n🏭 PRODUCT-WISE TEMPORAL SPLIT")
    print(f"{'='*70}")
    print(f"  Total products: {len(products)}")
    print(f"  Train products: {len(train_products)} ({len(train_products)/len(products)*100:.1f}%)")
    print(f"  Test products: {len(test_products)} ({len(test_products)/len(products)*100:.1f}%)")
    print(f"\n  Train samples: {len(train_df):,}")
    print(f"  Test samples: {len(test_df):,}")
    print(f"\n  ✅ Test set contains completely new products")
    
    return train_df, test_df

# Example usage (commented out):
# if 'timestamp' in df_processed.columns:
#     train_df, test_df = temporal_train_test_split(df_processed, 'timestamp')
# else:
#     print("⚠️  Timestamp column not found. Using product-wise split instead.")
#     train_df, test_df = temporal_train_test_split_by_product(df_processed)

print("\n✓ Temporal split functions defined")
print("  Two approaches available:")
print("    1. Time-based split (train on past, test on future)")
print("    2. Product-based split (train on old products, test on new ones)")


✓ Temporal split functions defined
  Two approaches available:
    1. Time-based split (train on past, test on future)
    2. Product-based split (train on old products, test on new ones)


In [39]:
# ========================================================================
# ADASYN & CLASS WEIGHT ALTERNATIVES (Better than SMOTE for Time-Series)
# ========================================================================

from imblearn.over_sampling import ADASYN
from sklearn.utils.class_weight import compute_class_weight

def balance_data_with_adasyn(X, y, categorical_features=None, random_state=42):
    """
    Balance dataset using ADASYN (Adaptive Synthetic Sampling).
    
    Why ADASYN instead of SMOTE for time-series:
    - ADASYN focuses on hard-to-learn minority samples (near decision boundary)
    - Less aggressive oversampling than SMOTE
    - Better preserves temporal structure
    
    But still not perfect for sequences! Consider class_weight instead.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix
    y : array-like
        Target labels
    categorical_features : list
        Indices of categorical features (not used in ADASYN)
    random_state : int
        Random seed
        
    Returns:
    --------
    X_resampled, y_resampled : tuple
        Balanced feature and target arrays
    """
    
    try:
        adasyn = ADASYN(random_state=random_state, n_neighbors=5)
        X_resampled, y_resampled = adasyn.fit_resample(X, y)
        
        print(f"\n🔄 ADASYN RESAMPLING")
        print(f"{'='*70}")
        print(f"  Original distribution:")
        for cls, count in zip(*np.unique(y, return_counts=True)):
            print(f"    Class {cls}: {count:,} samples ({count/len(y)*100:.1f}%)")
        
        print(f"\n  After ADASYN:")
        for cls, count in zip(*np.unique(y_resampled, return_counts=True)):
            print(f"    Class {cls}: {count:,} samples ({count/len(y_resampled)*100:.1f}%)")
        
        print(f"\n  ✅ Data balanced using adaptive synthetic sampling")
        
        return X_resampled, y_resampled
    
    except Exception as e:
        print(f"\n⚠️  ADASYN failed: {e}")
        print(f"  Returning original data without resampling")
        return X, y


def get_class_weights(y):
    """
    Calculate class weights for imbalanced data (RECOMMENDED for time-series).
    
    Why class weights > SMOTE/ADASYN for sequences:
    - No synthetic data generation (preserves temporal structure)
    - Forces model to pay more attention to minority class
    - Works natively with XGBoost (scale_pos_weight) and LSTM (class_weight)
    
    This is the BEST approach for time-series classification!
    
    Parameters:
    -----------
    y : array-like
        Target labels
        
    Returns:
    --------
    class_weight_dict : dict
        Dictionary mapping class labels to weights
    scale_pos_weight : float
        Weight ratio for binary classification (for XGBoost)
    """
    
    classes = np.unique(y)
    class_weights = compute_class_weight('balanced', classes=classes, y=y)
    class_weight_dict = dict(zip(classes, class_weights))
    
    print(f"\n⚖️  CLASS WEIGHTS CALCULATION")
    print(f"{'='*70}")
    print(f"  Class distribution:")
    for cls, count in zip(*np.unique(y, return_counts=True)):
        weight = class_weight_dict[cls]
        print(f"    Class {cls}: {count:,} samples ({count/len(y)*100:.1f}%) → Weight: {weight:.3f}")
    
    # For binary classification (XGBoost scale_pos_weight)
    if len(classes) == 2:
        neg_count = np.sum(y == 0)
        pos_count = np.sum(y == 1)
        scale_pos_weight = neg_count / pos_count
        print(f"\n  Binary classification:")
        print(f"    scale_pos_weight = {scale_pos_weight:.3f} (for XGBoost)")
        print(f"    Use: xgb.XGBClassifier(scale_pos_weight={scale_pos_weight:.3f})")
    else:
        scale_pos_weight = None
    
    print(f"\n  ✅ Class weights calculated (NO synthetic data generated)")
    print(f"  📝 Usage:")
    print(f"     - XGBoost: Pass as 'sample_weight' or 'scale_pos_weight'")
    print(f"     - LSTM: Pass as 'class_weight' in model.fit()")
    
    return class_weight_dict, scale_pos_weight


def apply_sample_weights(y, class_weight_dict):
    """
    Convert class weights to sample weights for training.
    
    Parameters:
    -----------
    y : array-like
        Target labels
    class_weight_dict : dict
        Class weight dictionary from get_class_weights()
        
    Returns:
    --------
    sample_weights : array
        Weight for each sample
    """
    sample_weights = np.array([class_weight_dict[label] for label in y])
    return sample_weights


# Comparison summary
print("\n" + "="*70)
print("📊 COMPARISON: SMOTE vs ADASYN vs CLASS WEIGHTS")
print("="*70)
print("""
Method          | Pros                              | Cons
----------------|-----------------------------------|----------------------------------
SMOTE           | Simple, widely used               | Breaks temporal structure
                | Balances all classes equally      | Creates unrealistic samples
                |                                   | May cause overfitting
                
ADASYN          | Focuses on hard samples           | Still generates synthetic data
                | Adaptive sampling                 | More complex than SMOTE
                | Better than SMOTE                 | Still breaks temporal patterns
                
CLASS WEIGHTS   | ✅ NO synthetic data              | Requires model support
(RECOMMENDED)   | ✅ Preserves temporal structure   | May need hyperparameter tuning
                | ✅ Works with sequences           | Less dramatic than oversampling
                | ✅ Production-ready               | 
                
RECOMMENDATION for Predictive Maintenance:
1. FIRST CHOICE: Use class_weight (preserves time-series nature)
2. SECOND CHOICE: ADASYN (if class_weight doesn't work)
3. LAST RESORT: SMOTE (only for tabular features, not sequences)
""")


📊 COMPARISON: SMOTE vs ADASYN vs CLASS WEIGHTS

Method          | Pros                              | Cons
----------------|-----------------------------------|----------------------------------
SMOTE           | Simple, widely used               | Breaks temporal structure
                | Balances all classes equally      | Creates unrealistic samples
                |                                   | May cause overfitting

ADASYN          | Focuses on hard samples           | Still generates synthetic data
                | Adaptive sampling                 | More complex than SMOTE
                | Better than SMOTE                 | Still breaks temporal patterns

CLASS WEIGHTS   | ✅ NO synthetic data              | Requires model support
(RECOMMENDED)   | ✅ Preserves temporal structure   | May need hyperparameter tuning
                | ✅ Works with sequences           | Less dramatic than oversampling
                | ✅ Production-ready               | 

RECOMMENDATION f

In [40]:
# ========================================================================
# TIME-SERIES CROSS-VALIDATION (More Reliable Performance Estimates)
# ========================================================================

from sklearn.model_selection import TimeSeriesSplit

def time_series_cross_validate(X, y, model, n_splits=5, metric_func=None):
    """
    Perform time-series cross-validation (rolling window).
    
    Why this matters:
    - Normal K-Fold: Random splits (FUTURE DATA LEAKAGE!)
    - TimeSeriesSplit: Expanding window (REALISTIC!)
    
    How it works:
        Fold 1: [Train: 1-20%]  [Test: 21-40%]
        Fold 2: [Train: 1-40%]  [Test: 41-60%]
        Fold 3: [Train: 1-60%]  [Test: 61-80%]
        Fold 4: [Train: 1-80%]  [Test: 81-100%]
    
    Each fold trains on past data and tests on future data.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix (should be time-ordered)
    y : array-like
        Target labels (should be time-ordered)
    model : estimator
        Scikit-learn compatible model
    n_splits : int
        Number of folds (default 5)
    metric_func : callable
        Scoring function (e.g., accuracy_score, f1_score)
        If None, uses model.score()
        
    Returns:
    --------
    scores : list
        Score for each fold
    avg_score : float
        Average score across folds
    std_score : float
        Standard deviation of scores
    """
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    
    print(f"\n⏳ TIME-SERIES CROSS-VALIDATION ({n_splits} folds)")
    print(f"{'='*70}")
    
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Train model
        model_clone = clone(model)  # Clone to avoid interference between folds
        model_clone.fit(X_train, y_train)
        
        # Evaluate
        if metric_func:
            y_pred = model_clone.predict(X_test)
            score = metric_func(y_test, y_pred)
        else:
            score = model_clone.score(X_test, y_test)
        
        scores.append(score)
        
        print(f"  Fold {fold_idx}:")
        print(f"    Train: {len(train_idx):,} samples ({train_idx[0]}-{train_idx[-1]})")
        print(f"    Test:  {len(test_idx):,} samples ({test_idx[0]}-{test_idx[-1]})")
        print(f"    Score: {score:.4f}")
    
    avg_score = np.mean(scores)
    std_score = np.std(scores)
    
    print(f"\n  {'─'*70}")
    print(f"  📊 CROSS-VALIDATION RESULTS:")
    print(f"    Average Score: {avg_score:.4f} ± {std_score:.4f}")
    print(f"    Min Score: {np.min(scores):.4f}")
    print(f"    Max Score: {np.max(scores):.4f}")
    print(f"    Score Range: {np.max(scores) - np.min(scores):.4f}")
    
    if std_score > 0.1:
        print(f"\n  ⚠️  High variance detected (std > 0.1)")
        print(f"     Model performance may be unstable over time")
    else:
        print(f"\n  ✅ Low variance - stable performance across time periods")
    
    return scores, avg_score, std_score


# Alternative: Walk-forward validation (most realistic for production)
def walk_forward_validate(df, time_col, target_col, feature_cols, 
                          model, n_splits=5, retrain_frequency=1):
    """
    Walk-forward validation (simulate production deployment).
    
    Most realistic validation approach:
    - Train on historical data
    - Predict next time window
    - Optionally retrain with new data
    - Move forward in time
    
    This mimics how model would perform in production!
    
    Parameters:
    -----------
    df : DataFrame
        Time-ordered dataframe
    time_col : str
        Timestamp column
    target_col : str
        Target column
    feature_cols : list
        Feature column names
    model : estimator
        Model to validate
    n_splits : int
        Number of time windows
    retrain_frequency : int
        Retrain every N windows (1 = retrain each window, 2 = every 2 windows)
        
    Returns:
    --------
    results : dict
        Comprehensive validation results
    """
    
    # Sort by time
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    
    # Split into time windows
    window_size = len(df_sorted) // (n_splits + 1)
    
    scores = []
    predictions_log = []
    
    print(f"\n🚶 WALK-FORWARD VALIDATION")
    print(f"{'='*70}")
    print(f"  Total samples: {len(df_sorted):,}")
    print(f"  Window size: {window_size:,}")
    print(f"  Retrain frequency: Every {retrain_frequency} window(s)")
    print(f"")
    
    # Initial training
    initial_end = window_size
    X_train = df_sorted[feature_cols].iloc[:initial_end].values
    y_train = df_sorted[target_col].iloc[:initial_end].values
    model.fit(X_train, y_train)
    
    # Walk forward
    for i in range(n_splits):
        test_start = (i + 1) * window_size
        test_end = (i + 2) * window_size
        
        # Test window
        X_test = df_sorted[feature_cols].iloc[test_start:test_end].values
        y_test = df_sorted[target_col].iloc[test_start:test_end].values
        
        # Predict
        y_pred = model.predict(X_test)
        score = accuracy_score(y_test, y_pred)
        scores.append(score)
        
        test_time_start = df_sorted[time_col].iloc[test_start]
        test_time_end = df_sorted[time_col].iloc[test_end-1]
        
        print(f"  Window {i+1}:")
        print(f"    Time: {test_time_start} to {test_time_end}")
        print(f"    Test samples: {len(y_test):,}")
        print(f"    Accuracy: {score:.4f}")
        
        # Log predictions
        predictions_log.append({
            'window': i+1,
            'time_start': test_time_start,
            'time_end': test_time_end,
            'n_samples': len(y_test),
            'accuracy': score,
            'y_true': y_test,
            'y_pred': y_pred
        })
        
        # Retrain if necessary
        if (i + 1) % retrain_frequency == 0:
            train_end = test_end
            X_train = df_sorted[feature_cols].iloc[:train_end].values
            y_train = df_sorted[target_col].iloc[:train_end].values
            model.fit(X_train, y_train)
            print(f"    🔄 Model retrained (train size: {len(y_train):,})")
        
    avg_score = np.mean(scores)
    std_score = np.std(scores)
    
    print(f"\n  {'─'*70}")
    print(f"  📊 WALK-FORWARD VALIDATION RESULTS:")
    print(f"    Average Accuracy: {avg_score:.4f} ± {std_score:.4f}")
    print(f"    This simulates production performance!")
    
    return {
        'scores': scores,
        'avg_score': avg_score,
        'std_score': std_score,
        'predictions_log': predictions_log
    }


print("\n✓ Time-series cross-validation functions defined")
print("  Two validation strategies:")
print("    1. TimeSeriesSplit: Expanding window cross-validation")
print("    2. Walk-Forward: Simulate production deployment (MOST REALISTIC)")


✓ Time-series cross-validation functions defined
  Two validation strategies:
    1. TimeSeriesSplit: Expanding window cross-validation
    2. Walk-Forward: Simulate production deployment (MOST REALISTIC)


In [41]:
# ========================================================================
# DATA QUALITY & VALIDATION CHECKS
# ========================================================================

def validate_predictive_maintenance_data(df, config=None):
    """
    Comprehensive data quality checks for predictive maintenance pipeline.
    
    Validates:
    - Missing values
    - Temporal ordering
    - Duplicate timestamps per product
    - Feature distributions (outliers, drift)
    - Target class balance
    - Data leakage indicators
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe
    config : dict (optional)
        Configuration with expected columns:
        {
            'product_id_col': 'product_id',
            'time_col': 'timestamp',
            'target_col': 'failure_type',
            'sensor_cols': ['air_temp', 'process_temp', 'rpm', 'torque_nm', 'tool_wear']
        }
        
    Returns:
    --------
    validation_report : dict
        Comprehensive validation results
    """
    
    if config is None:
        config = {
            'product_id_col': 'product_id',
            'time_col': 'timestamp',
            'target_col': 'failure_type',
            'sensor_cols': ['air_temp', 'process_temp', 'rpm', 'torque_nm', 'tool_wear']
        }
    
    print(f"\n🔍 DATA QUALITY VALIDATION")
    print(f"{'='*70}")
    
    report = {
        'passed': True,
        'warnings': [],
        'errors': [],
        'checks': {}
    }
    
    # ===== CHECK 1: Missing Values =====
    print(f"\n1️⃣  MISSING VALUES CHECK")
    missing_counts = df.isnull().sum()
    missing_pct = (missing_counts / len(df) * 100).round(2)
    
    for col, count in missing_counts[missing_counts > 0].items():
        pct = missing_pct[col]
        print(f"    ⚠️  {col}: {count:,} missing ({pct}%)")
        report['warnings'].append(f"{col} has {count} missing values ({pct}%)")
        
        if pct > 5:
            report['errors'].append(f"{col} missing > 5% - requires imputation")
            report['passed'] = False
    
    if missing_counts.sum() == 0:
        print(f"    ✅ No missing values")
    
    report['checks']['missing_values'] = missing_counts.to_dict()
    
    # ===== CHECK 2: Temporal Ordering =====
    if config['time_col'] in df.columns:
        print(f"\n2️⃣  TEMPORAL ORDERING CHECK")
        
        if not pd.api.types.is_datetime64_any_dtype(df[config['time_col']]):
            print(f"    ⚠️  {config['time_col']} is not datetime type")
            report['warnings'].append(f"{config['time_col']} should be datetime")
        
        # Check if sorted
        is_sorted = df[config['time_col']].is_monotonic_increasing
        if is_sorted:
            print(f"    ✅ Data is time-ordered")
        else:
            print(f"    ⚠️  Data is NOT time-ordered (needs sorting)")
            report['warnings'].append("Data not time-ordered")
        
        report['checks']['temporal_ordering'] = is_sorted
    
    # ===== CHECK 3: Duplicate Timestamps =====
    if config['product_id_col'] in df.columns and config['time_col'] in df.columns:
        print(f"\n3️⃣  DUPLICATE TIMESTAMPS CHECK")
        
        duplicates = df.groupby([config['product_id_col'], config['time_col']]).size()
        n_duplicates = (duplicates > 1).sum()
        
        if n_duplicates > 0:
            print(f"    ⚠️  {n_duplicates} product-timestamp combinations have duplicates")
            report['warnings'].append(f"{n_duplicates} duplicate timestamps per product")
        else:
            print(f"    ✅ No duplicate timestamps per product")
        
        report['checks']['duplicate_timestamps'] = n_duplicates
    
    # ===== CHECK 4: Target Class Balance =====
    if config['target_col'] in df.columns:
        print(f"\n4️⃣  TARGET CLASS BALANCE CHECK")
        
        class_counts = df[config['target_col']].value_counts()
        class_pcts = (class_counts / len(df) * 100).round(2)
        
        print(f"    Class distribution:")
        for cls, count in class_counts.items():
            pct = class_pcts[cls]
            print(f"      {cls}: {count:,} ({pct}%)")
        
        # Check for severe imbalance
        min_class_pct = class_pcts.min()
        if min_class_pct < 1:
            print(f"    ⚠️  Severe imbalance: Minority class < 1%")
            report['warnings'].append(f"Minority class only {min_class_pct}%")
        
        report['checks']['class_balance'] = class_counts.to_dict()
    
    # ===== CHECK 5: Feature Distribution (Outliers) =====
    print(f"\n5️⃣  FEATURE DISTRIBUTION CHECK")
    
    for col in config['sensor_cols']:
        if col not in df.columns:
            continue
            
        # IQR method for outlier detection
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 3 * IQR
        upper_bound = Q3 + 3 * IQR
        
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        outlier_pct = (outliers / len(df) * 100).round(2)
        
        if outlier_pct > 5:
            print(f"    ⚠️  {col}: {outliers:,} outliers ({outlier_pct}%)")
            report['warnings'].append(f"{col} has {outlier_pct}% outliers")
    
    print(f"    ✅ Feature distribution checked")
    
    # ===== CHECK 6: Data Leakage Indicators =====
    print(f"\n6️⃣  DATA LEAKAGE CHECK")
    
    # Check for perfect correlations (potential leakage)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    corr_matrix = df[numeric_cols].corr().abs()
    
    # Find near-perfect correlations (excluding diagonal)
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] > 0.95:
                col1 = corr_matrix.columns[i]
                col2 = corr_matrix.columns[j]
                corr_val = corr_matrix.iloc[i, j]
                high_corr_pairs.append((col1, col2, corr_val))
    
    if high_corr_pairs:
        print(f"    ⚠️  HIGH CORRELATIONS DETECTED (potential leakage):")
        for col1, col2, corr in high_corr_pairs:
            print(f"       {col1} ↔ {col2}: {corr:.4f}")
        report['warnings'].append(f"{len(high_corr_pairs)} high correlation pairs found")
    else:
        print(f"    ✅ No suspicious correlations detected")
    
    report['checks']['high_correlations'] = high_corr_pairs
    
    # ===== FINAL REPORT =====
    print(f"\n{'='*70}")
    print(f"📋 VALIDATION SUMMARY:")
    print(f"  Status: {'✅ PASSED' if report['passed'] else '❌ FAILED'}")
    print(f"  Warnings: {len(report['warnings'])}")
    print(f"  Errors: {len(report['errors'])}")
    
    if report['errors']:
        print(f"\n  ❌ CRITICAL ERRORS:")
        for error in report['errors']:
            print(f"     - {error}")
    
    if report['warnings']:
        print(f"\n  ⚠️  WARNINGS:")
        for warning in report['warnings'][:5]:  # Show first 5
            print(f"     - {warning}")
        if len(report['warnings']) > 5:
            print(f"     ... and {len(report['warnings']) - 5} more")
    
    print(f"{'='*70}")
    
    return report


print("\n✓ Data validation function defined")
print("  Run validate_predictive_maintenance_data(df) to check data quality")


✓ Data validation function defined
  Run validate_predictive_maintenance_data(df) to check data quality
